# Day 3: Exploratory Data Analysis (EDA)

In this notebook, we connect to our locally built SQLite Star Schema database (`bluestock_mf.db`) to uncover data-driven insights about the Indian Mutual Fund industry, scheme performance, and investor demographics.

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Set visual aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Connect to Database
BASE_DIR = Path().resolve().parent
DB_PATH = BASE_DIR / 'data' / 'db' / 'bluestock_mf.db'
conn = sqlite3.connect(DB_PATH)
print(f"Successfully connected to Database: {DB_PATH}")

## 1. Macro Industry Trends
### Total AUM by Fund House
Let's visualize the latest AUM (Assets Under Management) footprint for the top AMCs in our dataset.

In [ ]:
query = """
SELECT fund_house, SUM(aum_crore) as total_aum_cr
FROM fact_aum
GROUP BY fund_house
ORDER BY total_aum_cr DESC
LIMIT 10;
"""
df_aum = pd.read_sql(query, conn)

fig = px.bar(df_aum, x='fund_house', y='total_aum_cr', 
             title='Top 10 Fund Houses by Total AUM (Crores)',
             labels={'fund_house': 'AMC', 'total_aum_cr': 'AUM (₹ Crores)'},
             color='total_aum_cr', color_continuous_scale='Blues')
fig.show()

**Insight**: The Indian MF market is highly consolidated at the top, with a steep drop-off in AUM after the top 3-4 players (like SBI, ICICI, and HDFC).

## 2. Fund Performance & Expense Analysis
Does a higher Expense Ratio guarantee better returns? Let's plot the 5-Year Return vs Expense Ratio.

In [ ]:
query = """
SELECT p.amfi_code, f.scheme_name, f.category, f.expense_ratio_pct, p.return_5yr_pct, p.aum_crore
FROM fact_performance p
JOIN dim_fund f ON p.amfi_code = f.amfi_code
WHERE p.return_5yr_pct IS NOT NULL
"""
df_perf = pd.read_sql(query, conn)

fig = px.scatter(df_perf, x='expense_ratio_pct', y='return_5yr_pct', 
                 color='category', size='aum_crore', hover_name='scheme_name',
                 title='5-Year Returns vs Expense Ratio (Bubble Size = AUM)',
                 labels={'expense_ratio_pct': 'Expense Ratio (%)', 'return_5yr_pct': '5Yr Return (%)'})
fig.add_hline(y=df_perf['return_5yr_pct'].mean(), line_dash="dot", annotation_text="Average Return")
fig.show()

**Insight**: 
- Interestingly, the funds charging the absolute highest expense ratios do *not* uniformly generate the highest returns. 
- The largest "bubbles" (high AUM funds) are clustered towards the left, indicating that top funds achieve scale while maintaining competitive (lower) expense ratios.

## 3. Investor Demographics & Transactions
Understanding who is investing and where the money is coming from.

In [ ]:
query_tx = """
SELECT transaction_type, SUM(amount_inr) as total_value, COUNT(transaction_id) as total_volume
FROM fact_transactions
GROUP BY transaction_type;
"""
df_tx = pd.read_sql(query_tx, conn)

# Create 1x2 subplots using matplotlib for variety
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].pie(df_tx['total_value'], labels=df_tx['transaction_type'], autopct='%1.1f%%', colors=sns.color_palette('pastel'))
axes[0].set_title('Transaction Breakdown by Total Value (₹)')

axes[1].pie(df_tx['total_volume'], labels=df_tx['transaction_type'], autopct='%1.1f%%', colors=sns.color_palette('Set2'))
axes[1].set_title('Transaction Breakdown by Volume (Count)')

plt.tight_layout()
plt.show()

**Insight**: While SIPs likely dominate the total *volume* (number of transactions), LUMPSUMs constitute a massive portion of the total *value* flowing in.

In [ ]:
query_state = """
SELECT state, SUM(amount_inr)/10000000 as total_cr
FROM fact_transactions
GROUP BY state
ORDER BY total_cr DESC
LIMIT 10;
"""
df_state = pd.read_sql(query_state, conn)

plt.figure(figsize=(12, 6))
sns.barplot(data=df_state, x='total_cr', y='state', hue='state', palette='viridis', legend=False)
plt.title('Top 10 States by Total Investment (₹ Crores)')
plt.xlabel('Total Investment (Crores)')
plt.ylabel('State')
plt.show()

## 4. Portfolio Allocations
Where is the Mutual Fund industry deploying capital?

In [ ]:
# Because portfolio_holdings is not explicitly in schema, 
# we loaded it directly as 'portfolio_holdings' table in SQLite during Day 2.
try:
    query_port = """
    SELECT sector, SUM(market_value_cr) as sector_mv
    FROM portfolio_holdings
    GROUP BY sector
    ORDER BY sector_mv DESC
    LIMIT 10;
    """
    df_port = pd.read_sql(query_port, conn)
    
    fig = px.treemap(df_port, path=['sector'], values='sector_mv',
                     title='Top 10 Sectors by Mutual Fund Allocation (Treemap)',
                     color='sector_mv', color_continuous_scale='Teal')
    fig.show()
except sqlite3.OperationalError:
    print("Table 'portfolio_holdings' not fully structured in star schema yet, skipping.")

In [ ]:
# Close the database connection
conn.close()